In [1]:
import sys
print(sys.executable)

/venv/main/bin/python


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])


In [2]:
from sklearn import show_versions
show_versions()


System:
    python: 3.12.15 (main, Oct 13 2026, 16:52:04) [GCC 15.3.0]
executable: /venv/main/bin/python
   machine: Linux-5.15.0-202-generic-x86_64-with-glibc2.39

Python dependencies:
      sklearn: 1.9.1
          pip: 26.2.2
   setuptools: 84.2.1
        numpy: 2.5.4
        scipy: 1.18.1
       Cython: None
       pandas: 3.1.1
   matplotlib: 3.11.2
       joblib: 1.6.0
threadpoolctl: 3.8.0
     narwhals: 2.28.0

Built with OpenMP: True

threadpoolctl info:
       user_api: blas
   internal_api: openblas
    num_threads: 24
         prefix: libscipy_openblas
       filepath: /venv/main/lib/python3.12/site-packages/scipy.libs/libscipy_openblas-5f890258.so
        version: 0.3.31.dev
threading_layer: pthreads
   architecture: Haswell

       user_api: blas
   internal_api: openblas
    num_threads: 24
         prefix: libscipy_openblas
       filepath: /venv/main/lib/python3.12/site-packages/numpy.libs/libscipy_openblas64_-f48b354e.so
        version: 0.3.34.106.0
threading_layer: 

In [4]:
import os
from pathlib import Path
import glob
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedGroupKFold
from imblearn.metrics import geometric_mean_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import time
sys.path.insert(0, os.getcwd())
import model_lib
import importlib
import json
from run_utils import start_run, save_json, save_predictions, sha256
from datetime import datetime


In [ ]:
# Prepare full data once: python preprocessing.py
# In a notebook: subprocess.check_call([sys.executable, "preprocessing.py"])
# Skip preparation when dataset/sleep_waveform.pt already exists.


In [5]:
class EEGDataset(Dataset):
    def __init__(self, pt_filepath, split="development"):
        """
        Args:
            pt_filepath (str): Saved (N, 2, 30, 200) Sleep-EDF waveform patches.
            split (str): "development" for CV, or "test" for held-out evaluation.
        """
        print(f"Memory-mapping dataset from {pt_filepath}...")
        # Tensors and ordinary string/list metadata support weights_only=True.
        checkpoint = torch.load(pt_filepath, map_location="cpu", weights_only=True, mmap=True)

        self.data = checkpoint['patches']
        # Select one participant partition without copying the waveform tensor.
        self.indices = np.flatnonzero(np.asarray(checkpoint['splits']) == split)
        self.labels = checkpoint['labels'][self.indices]
        self.groups = np.asarray(checkpoint['groups'])[self.indices]
        self.validation_folds = checkpoint['validation_folds'][self.indices].numpy()
        self.recordings = np.asarray(checkpoint['recordings'])[self.indices]
        self.onsets = checkpoint['onsets'][self.indices].numpy()

        self.channels = checkpoint['channels']
        self.sfreq = checkpoint['sfreq']
        self.amplitude = checkpoint['preprocessing']['amplitude']
        self.class_to_idx = checkpoint['class_map']
        self.idx_to_class = {value: key for key, value in self.class_to_idx.items()}

        print("Mapped source shape:", tuple(self.data.shape))
        print("Input per epoch:", tuple(self.data.shape[1:]), "= channels, seconds, samples/second")
        print("Channels:", self.channels, "| Sampling rate:", self.sfreq, "Hz | Scale:", self.amplitude)
        print(f"{split.capitalize()} samples:", len(self.labels))
        print("Participants:", len(set(self.groups)))
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Preserve (channels, 30 one-second patches, 200 samples) and stored amplitude.
        x = self.data[self.indices[idx]].float()
        y = self.labels[idx].long()

        return x, y

def get_stats(subset):
    """Diagnostic mean/std per EEG channel, across epochs and waveform samples.

    Input batches: (B, C, 30, 200). Flatten only the two time axes to (B, C, 6000).
    These statistics are not applied to the pretrained CBraMod input: the export
    already uses microvolts / 100. Call on a training subset when inspecting data.
    """
    loader = DataLoader(subset, batch_size=32, shuffle=False, num_workers=0)
    sum_ = torch.zeros(subset[0][0].shape[0], dtype=torch.float64)
    sum_sq = torch.zeros_like(sum_)
    total_samples = 0

    for waveforms, _ in loader:
        waveforms = waveforms.double().flatten(start_dim=2)
        sum_ += waveforms.sum(dim=(0, 2))
        sum_sq += waveforms.square().sum(dim=(0, 2))
        total_samples += waveforms.shape[0] * waveforms.shape[2]

    mean = sum_ / total_samples
    variance = sum_sq / total_samples - mean.square()
    std = variance.clamp_min(0).sqrt()
    return mean.float(), std.float()

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    def __call__(self, val_f1):
        if val_f1 > self.best_f1 + self.min_delta:
            self.best_f1 = val_f1
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
class ApplyTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

def calculate_metrics(targets, outputs):
    """Five-stage metrics from epoch-level argmax predictions.

    MGm = mean(sqrt(specificity_i * recall_i)) across the five stages,
    matching the paper: compute per-class G-means first, then average.
    Per-class F1 order: W, N1, N2, N3, REM; undefined class ratios use zero.
    """
    outputs = outputs.detach().float().cpu()
    targets = targets.view(-1).detach().cpu().numpy().astype(int)
    preds = outputs.argmax(dim=1).numpy()

    acc = accuracy_score(targets, preds)
    bal_acc = balanced_accuracy_score(targets, preds)
    prec = precision_score(targets, preds, labels=range(5), average='macro', zero_division=0)
    rec = recall_score(targets, preds, labels=range(5), average='macro', zero_division=0)
    per_class_f1 = f1_score(targets, preds, labels=range(5), average=None, zero_division=0)
    f1 = per_class_f1.mean()
    kappa = cohen_kappa_score(targets, preds, labels=range(5))

    mgm = geometric_mean_score(targets, preds, labels=range(5), average=None).mean()

    return acc, mgm, prec, rec, bal_acc, f1, kappa, per_class_f1

def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, scheduler, device, use_sched, use_es, max_epochs=50, checkpoint_path=None):
    # Development/inner validation only: this function selects an epoch using val_loader.
    scaler = GradScaler('cuda', enabled=device.type == 'cuda')
    early_stopping = EarlyStopping(patience=10, min_delta=0.001)

    # Select the checkpoint and stop training using validation Macro-F1.
    best_mgm = 0.0
    best_acc = 0.0
    best_prec = 0.0
    best_rec = 0.0
    best_bal_acc = 0.0
    best_f1 = -float('inf')
    best_kappa = 0.0
    best_per_class_f1 = np.zeros(5)
    best_epoch, best_state = 0, None
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    history = []
    for epoch in range(max_epochs):
        epoch_lr = optimizer.param_groups[0]["lr"]
        epoch_start_time = time.time()

        model.train()
        train_loss = 0.0
        train_weight_sum = 0.0

        all_train_targets, all_train_outputs = [], []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device=device, dtype=torch.float32), labels.to(device=device, dtype=torch.long)

            optimizer.zero_grad()

            with autocast('cuda', enabled=device.type == 'cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_weight = criterion.weight[labels].sum().item() if criterion.weight is not None else len(labels)
            train_loss += loss.item() * batch_weight
            train_weight_sum += batch_weight

            all_train_targets.append(labels.detach().cpu())
            all_train_outputs.append(outputs.detach().cpu())

        model.eval()
        val_loss = 0.0
        val_weight_sum = 0.0
        all_targets, all_outputs = [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device=device, dtype=torch.float32), labels.to(device=device, dtype=torch.long)

                with autocast('cuda', enabled=device.type == 'cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                batch_weight = criterion.weight[labels].sum().item() if criterion.weight is not None else len(labels)
                val_loss += loss.item() * batch_weight
                val_weight_sum += batch_weight
                all_targets.append(labels.cpu())
                all_outputs.append(outputs.cpu())

        avg_train_loss = train_loss / train_weight_sum
        avg_val_loss = val_loss / val_weight_sum

        train_targets_tensor = torch.cat(all_train_targets)
        train_outputs_tensor = torch.cat(all_train_outputs)
        train_acc, train_mgm, train_prec, train_rec, train_bal_acc, train_f1, train_kappa, _ = calculate_metrics(train_targets_tensor, train_outputs_tensor)

        targets_tensor = torch.cat(all_targets)
        outputs_tensor = torch.cat(all_outputs)
        acc, mgm, prec, rec, bal_acc, f1, kappa, per_class_f1 = calculate_metrics(targets_tensor, outputs_tensor)
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(acc)

        epoch_duration = time.time() - epoch_start_time

        print(f"Epoch {epoch+1:02d}/{max_epochs} | Time: {epoch_duration:.2f}s")
        print(f"  [Train] Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | BAcc: {train_bal_acc:.4f} | MGm: {train_mgm:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | Kappa: {train_kappa:.4f}")
        print(f"  [Val]   Loss: {avg_val_loss:.4f} | Acc: {acc:.4f} | BAcc: {bal_acc:.4f} | MGm: {mgm:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Kappa: {kappa:.4f}")

        row = {"epoch": epoch + 1, "seconds": epoch_duration, "lr": epoch_lr,
               "train_loss": avg_train_loss, "val_loss": avg_val_loss}
        for prefix, values in [
            ("train", [train_acc, train_bal_acc, train_f1, train_kappa, train_mgm, train_prec, train_rec]),
            ("val", [acc, bal_acc, f1, kappa, mgm, prec, rec]),
        ]:
            row.update({f"{prefix}_{name}": float(value) for name, value in zip(
                ["acc", "bal_acc", "f1", "kappa", "mgm", "prec", "rec"], values)})
        row.update({f"val_f1_{stage}": float(value) for stage, value in zip(
            ['W', 'N1', 'N2', 'N3', 'REM'], per_class_f1)})
        history.append(row)
        if checkpoint_path is not None:
            pd.DataFrame(history).to_csv(checkpoint_path.with_suffix(".history.csv"), index=False)

        if use_sched:
            scheduler.step()

        if f1 > best_f1:
            best_acc = acc
            best_mgm = mgm
            best_prec = prec
            best_rec = rec
            best_bal_acc = bal_acc
            best_f1 = f1
            best_kappa = kappa
            best_per_class_f1 = per_class_f1.copy()
            best_epoch = epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_targets = targets_tensor.numpy().astype(int)
            best_preds = outputs_tensor.float().argmax(dim=1).numpy()
            if checkpoint_path is not None:
                torch.save(best_state, checkpoint_path)
                save_json(checkpoint_path.with_suffix(".best.json"), {
                    "selection_metric": "validation macro-F1", "best_epoch": best_epoch,
                    "metrics": row, "confusion_matrix": confusion_matrix(
                        best_targets, best_preds, labels=range(5)).tolist(),
                })

        if use_es:
            early_stopping(f1)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}!")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.eval()
        print(f'Restored best validation-Macro-F1 weights from epoch {best_epoch}.')

    if best_state is not None:
        cm = confusion_matrix(best_targets, best_preds, labels=range(5))
        print(f'Validation EEG-epoch confusion matrix at epoch {best_epoch} (argmax over five stages)')
        print('Rows = true [W, N1, N2, N3, REM]; columns = predicted stages')
        print(cm)
        ConfusionMatrixDisplay(cm, display_labels=['W', 'N1', 'N2', 'N3', 'REM']).plot(cmap='Blues', values_format='d', colorbar=False)
        plt.title(f'Validation (EEG epochs) - best Macro-F1 epoch {best_epoch}')
        plt.tight_layout()
        plt.show()

        epochs = range(1, len(train_losses) + 1)
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(epochs, train_losses, label='Train Loss')
        plt.plot(epochs, val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.subplot(1, 2, 2)
        plt.plot(epochs, train_accuracies, label='Train Accuracy')
        plt.plot(epochs, val_accuracies, label='Val Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.suptitle('EEG-epoch metrics; training measured during updates')
        plt.tight_layout()
        plt.show()

    return best_mgm, best_acc, best_prec, best_rec, best_bal_acc, best_f1, best_kappa, best_per_class_f1

In [8]:
# Open Jupyter in the directory containing model_lib.py (the contents of A2).
PROJECT_DIR = Path.cwd()
CONFIG_PATH = PROJECT_DIR / "configs" / "AttnSleep.json"  # Change only this filename.
settings = json.loads(CONFIG_PATH.read_text())
#DATA_PATH = PROJECT_DIR / settings["data_path"]
DATA_PATH = PROJECT_DIR / "sleep_waveform.pt"
CHECKPOINT = PROJECT_DIR / "pretrained_weights" / "pretrained_weights.pth"
print("Configuration:", CONFIG_PATH.name, "| Dataset:", DATA_PATH)


Configuration: AttnSleep.json | Dataset: /sleep_waveform.pt


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(settings["seed"])
np.random.seed(settings["seed"])

dataset = EEGDataset(pt_filepath=DATA_PATH)
print(f"Found {len(dataset)} total samples across {len(dataset.class_to_idx)} classes.")

Using device: cuda
Memory-mapping dataset from /dataset/sleep_waveform.pt...


FileNotFoundError: [Errno 2] No such file or directory: '/dataset/sleep_waveform.pt'

In [ ]:
importlib.reload(model_lib)

# One notebook, one selected JSON config. Optional overrides go before `config =`.
MODEL_NAME = settings["model_name"]
MODEL_KWARGS = dict(settings["model_kwargs"])
if "checkpoint_path" in MODEL_KWARGS:
    MODEL_KWARGS["checkpoint_path"] = str(PROJECT_DIR / MODEL_KWARGS["checkpoint_path"])
RUN_NAME = f"{Path(DATA_PATH).stem}_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_DIR = PROJECT_DIR / "runs" / MODEL_NAME / RUN_NAME
SCHEDULER_NAME = settings["scheduler_name"]
SCHEDULER_KWARGS = settings["scheduler_kwargs"]

lr = settings["lr"]
batch_size = settings["batch_size"]
weight_decay = settings["weight_decay"]
n_splits = settings["n_splits"]
max_epochs = settings["max_epochs"]

# Linux/CUDA defaults. For a macOS notebook edit the config to 0 / null / false / false.
num_workers = settings["num_workers"]
prefetch_factor = settings["prefetch_factor"]
persistent_workers = settings["persistent_workers"]
pin_memory = settings["pin_memory"]

# Reuse saved participant folds, identical across models and full/small exports.
assert set(dataset.validation_folds) == set(range(1, n_splits + 1))
y = dataset.labels.numpy()
groups = np.array(dataset.groups)
fold_splits = [(np.flatnonzero(dataset.validation_folds != i),
                np.flatnonzero(dataset.validation_folds == i))
               for i in range(1, n_splits + 1)]
assert tuple(dataset.data.shape[1:]) == (2, 30, 200)
assert dataset.sfreq == 200 and dataset.amplitude == "microvolts / 100"
for train_idx, val_idx in fold_splits:
    assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))

config = {
    "config_file": str(CONFIG_PATH.resolve()),
    "model_name": MODEL_NAME, "model_kwargs": MODEL_KWARGS,
    "data_path": str(Path(DATA_PATH).resolve()),
    "seed": settings["seed"], "lr": lr, "batch_size": batch_size, "weight_decay": weight_decay,
    "n_splits": n_splits, "max_epochs": max_epochs,
    "num_workers": num_workers, "prefetch_factor": prefetch_factor,
    "persistent_workers": persistent_workers, "pin_memory": pin_memory,
    "optimizer": "Adam", "amsgrad": True,
    "scheduler_name": SCHEDULER_NAME, "scheduler_kwargs": SCHEDULER_KWARGS,
    "loss": "weighted CrossEntropyLoss", "class_weight_mu": [0.3, 0.4, 0.3, 0.2, 0.3],
    "selection_metric": "validation macro-F1", "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.001, "amp": device.type == "cuda",
    "fold_policy": "saved participant validation_folds; test excluded",
}
run_metadata = start_run(RUN_DIR, config, dataset, device)
torch.manual_seed(settings["seed"])
np.random.seed(settings["seed"])

# Balanced Accuracy is Macro-Recall when all five stages are present.
cv_metrics = {
    'acc': [],
    'bal_acc': [],
    'f1': [],
    'kappa': [],
    'mgm': [],
    'prec': [],
    'rec': [],
}
cv_per_class_f1 = []
stage_names = ['W', 'N1', 'N2', 'N3', 'REM']

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    # if fold > 1: break  # Uncomment for a two-fold pilot; final comparison uses all five.
    print(f"\n{'='*20} Fold {fold + 1}/{n_splits} {'='*20}")

    train_subset_raw = Subset(dataset, train_idx)
    val_subset_raw = Subset(dataset, val_idx)

    # The saved waveform is already microvolts / 100. Do not z-score or divide again.
    # Optional diagnostics only: mean, std = get_stats(train_subset_raw)
    # Reference: https://github.com/wjq-learning/CBraMod/blob/main/datasets/isruc_dataset.py
    normalization = None

    train_subset = ApplyTransform(train_subset_raw, transform=normalization)
    val_subset = ApplyTransform(val_subset_raw, transform=normalization)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, prefetch_factor=prefetch_factor,
                                num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, prefetch_factor=prefetch_factor,
                                num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers)

    model = getattr(model_lib, MODEL_NAME)(**MODEL_KWARGS).to(device)
    # Author's class-weight formula (mu tuned for Sleep-EDF-20), reused here.
    # Count ONLY training labels; do not use validation/test labels for weights.
    # https://github.com/emadeldeen24/AttnSleep/blob/main/utils/util.py
    counts = np.bincount(y[train_idx], minlength=5)
    mu = np.array([1.5, 2.0, 1.5, 1.0, 1.5]) / 5
    weights = np.round(np.maximum(np.log(mu * counts.sum() / counts), 1.0) * mu, 2)
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    print("Class weights [W, N1, N2, N3, REM]:", weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay, amsgrad=True)
    scheduler = getattr(torch.optim.lr_scheduler, SCHEDULER_NAME)(optimizer, **SCHEDULER_KWARGS)
    checkpoint_path = RUN_DIR / "checkpoints" / f"fold_{fold + 1}.pt"
    (RUN_DIR / "architecture.txt").write_text(str(model) + "\n")
    save_json(checkpoint_path.with_suffix(".setup.json"), {
        "fold": fold + 1, "train_samples": len(train_idx), "val_samples": len(val_idx),
        "train_participants": sorted(set(groups[train_idx])),
        "val_participants": sorted(set(groups[val_idx])),
        "class_weights": weights.tolist(),
        "parameters": sum(p.numel() for p in model.parameters()),
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    })

    best_mgm, best_acc, best_prec, best_rec, best_bal_acc, best_f1, best_kappa, best_per_class_f1 = train_and_evaluate(
        model, train_loader, val_loader, criterion, optimizer, scheduler, device, use_sched=True, use_es=True, max_epochs=max_epochs, checkpoint_path=checkpoint_path
    )

    # train_and_evaluate has already restored the best validation checkpoint.
    run_metadata["completed_folds"].append({"fold": fold + 1,
        "checkpoint": str(checkpoint_path.relative_to(RUN_DIR)), "sha256": sha256(checkpoint_path)})
    save_json(RUN_DIR / "run.json", run_metadata)
    print("Saved best validation checkpoint:", checkpoint_path)

    cv_metrics['mgm'].append(best_mgm)
    cv_metrics['acc'].append(best_acc)
    cv_metrics['prec'].append(best_prec)
    cv_metrics['rec'].append(best_rec)
    cv_metrics['bal_acc'].append(best_bal_acc)
    cv_metrics['f1'].append(best_f1)
    cv_metrics['kappa'].append(best_kappa)
    cv_per_class_f1.append(best_per_class_f1)

    print(f"\n--- Fold {fold + 1} Metrics at Best Validation Macro-F1 ---")
    print(f"Acc: {best_acc:.4f} | BAcc: {best_bal_acc:.4f} | MGm: {best_mgm:.4f} | Prec: {best_prec:.4f} | Rec: {best_rec:.4f} | F1: {best_f1:.4f} | Kappa: {best_kappa:.4f}")

print(f"\n{'='*20} Final CV Results ({len(cv_metrics['acc'])}/{n_splits} folds completed) {'='*20}")
# All columns describe the same selected checkpoint in each fold.
cv_results = pd.DataFrame(cv_metrics, index=[f"Fold {i + 1}" for i in range(len(cv_metrics['acc']))])
cv_results.columns = pd.MultiIndex.from_tuples([
    ('Overall', name) for name in ['Accuracy', 'Balanced Accuracy', 'F1', 'Kappa', 'MGm', 'Precision', 'Recall']
])
per_class_results = pd.DataFrame(
    cv_per_class_f1, index=cv_results.index,
    columns=pd.MultiIndex.from_product([['Per-class F1'], stage_names]),
)
cv_results = pd.concat([cv_results, per_class_results], axis=1)
cv_summary = pd.DataFrame(
    [[f"{cv_results[col].mean():.4f} ± {cv_results[col].std(ddof=0):.4f}" for col in cv_results.columns]],
    columns=cv_results.columns, index=['Mean ± SD'],
)
display(cv_results.round(4), cv_summary)
cv_results.to_csv(RUN_DIR / "cv_results.csv")
cv_summary.to_csv(RUN_DIR / "cv_summary.csv")
pd.concat([
    pd.read_csv(RUN_DIR / "checkpoints" / f"fold_{i + 1}.history.csv").assign(fold=i + 1)
    for i in range(len(cv_results))
], ignore_index=True).to_csv(RUN_DIR / "history.csv", index=False)
run_metadata["status"] = "cv_complete" if len(cv_results) == n_splits else "partial_cv"
save_json(RUN_DIR / "run.json", run_metadata)


In [ ]:
# Pool validation predictions from the saved best checkpoint of EACH fold.
# Reuse the SAME dataset, SGKF splits, model architecture and input settings.
# No training; these are pooled validation results, not independent test results.
completed_folds = len(cv_results)
fold_checkpoints = [
    RUN_DIR / "checkpoints" / f"fold_{i + 1}.pt"
    for i in range(completed_folds)
]
assert completed_folds > 0 and all(p.is_file() for p in fold_checkpoints), "Missing fold checkpoints: per-fold metrics alone cannot recover predictions."

# Restore the current model afterward so the held-out test cell still uses it.
current_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
pooled_targets, pooled_outputs, pooled_indices = [], [], []
for fold, (_, val_idx) in enumerate(fold_splits):
    if fold >= completed_folds: break
    model.load_state_dict(torch.load(fold_checkpoints[fold], map_location="cpu", weights_only=True))
    model.eval()
    fold_val_loader = DataLoader(
        Subset(dataset, val_idx), batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=device.type == "cuda",
    )
    with torch.inference_mode():
        for inputs, labels in fold_val_loader:
            inputs = inputs.to(device=device, dtype=torch.float32, non_blocking=True)
            with autocast('cuda', enabled=device.type == 'cuda'):
                outputs = model(inputs)
            pooled_targets.append(labels.cpu())
            pooled_outputs.append(outputs.float().cpu())
    pooled_indices.append(val_idx)
    print(f"Collected Fold {fold + 1}: {len(val_idx)} validation samples")

model.load_state_dict(current_state)
model.eval()
del current_state
pooled_targets = torch.cat(pooled_targets)
pooled_outputs = torch.cat(pooled_outputs)
pooled_indices = np.concatenate(pooled_indices)
assert len(np.unique(pooled_indices)) == len(pooled_indices), "Duplicate validation samples."
assert np.array_equal(pooled_targets.numpy(), y[pooled_indices]), "Validation order does not match labels."
if completed_folds == n_splits:
    assert len(pooled_indices) == len(dataset), "Incomplete development coverage."

acc, mgm, prec, rec, bal_acc, f1, kappa, per_class_f1 = calculate_metrics(pooled_targets, pooled_outputs)
pooled_results = pd.DataFrame(
    [[acc, bal_acc, f1, kappa, mgm, prec, rec, *per_class_f1]],
    columns=cv_results.columns,
    index=[f"Pooled validation ({completed_folds}/{n_splits} folds)"],
)
print(f"Pooled validation: {len(pooled_targets)} EEG epochs; each included once")
display(pooled_results)

pooled_preds = pooled_outputs.argmax(dim=1).numpy()
pooled_cm = confusion_matrix(pooled_targets.numpy(), pooled_preds, labels=range(5))
print("Rows = true [W, N1, N2, N3, REM]; columns = predicted stages")
print(pooled_cm)
ConfusionMatrixDisplay(pooled_cm, display_labels=stage_names).plot(
    cmap='Blues', values_format='d', colorbar=False,
)
plt.title(f"Pooled validation - {completed_folds}/{n_splits} folds")
plt.tight_layout()
plt.show()
pooled_results.to_csv(RUN_DIR / "pooled_validation_metrics.csv")
np.savetxt(RUN_DIR / "pooled_validation_confusion_matrix.csv", pooled_cm, delimiter=",", fmt="%d")
save_predictions(RUN_DIR / "pooled_validation_predictions.csv", dataset, pooled_indices,
                 pooled_targets.numpy(), torch.softmax(pooled_outputs, dim=1),
                 folds=dataset.validation_folds[pooled_indices])


In [ ]:
# Equal-weight soft voting across all K best-validation fold checkpoints.
# Use the SAME DATA_PATH, architecture and input settings as training.
K = n_splits
fold_checkpoints = [
    RUN_DIR / "checkpoints" / f"fold_{fold + 1}.pt"
    for fold in range(K)
]
assert len(cv_results) == K, "Complete all K folds before evaluating the K-model ensemble."
assert all(p.is_file() for p in fold_checkpoints), "Missing best-validation fold checkpoints."

test_dataset = EEGDataset(pt_filepath=DATA_PATH, split="test")
assert set(dataset.groups).isdisjoint(set(test_dataset.groups))
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=0, pin_memory=device.type == "cuda",
)

current_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
test_probs_list, test_targets = [], []
for fold in range(K):
    model.load_state_dict(torch.load(fold_checkpoints[fold], map_location="cpu", weights_only=True))
    model.eval()
    fold_probs = []
    with torch.inference_mode():
        for inputs, labels in test_loader:
            inputs = inputs.to(device=device, dtype=torch.float32, non_blocking=True)
            with autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(inputs)
            fold_probs.append(torch.softmax(logits.float(), dim=1).cpu())
            if fold == 0:
                test_targets.append(labels.cpu())
    test_probs_list.append(torch.cat(fold_probs))
    print(f"Collected test probabilities from Fold {fold + 1}/{K}")

model.load_state_dict(current_state)
model.eval()
del current_state
# [K, N, 5] -> [N, 5]: average probabilities, then choose the final stage.
mean_test_probs = torch.stack(test_probs_list).mean(dim=0)
test_targets = torch.cat(test_targets)
assert torch.equal(test_targets, test_dataset.labels.cpu()), "Test label order mismatch."
final_test_preds = mean_test_probs.argmax(dim=1)
acc, mgm, prec, rec, bal_acc, f1, kappa, per_class_f1 = calculate_metrics(test_targets, mean_test_probs)
test_results = pd.DataFrame(
    [[acc, bal_acc, f1, kappa, mgm, prec, rec, *per_class_f1]],
    columns=cv_results.columns, index=[f"Held-out test ({K}-model soft-voting)"],
)
print(f"Held-out test: {len(test_dataset)} EEG epochs, {len(set(test_dataset.groups))} participants; {K}-model ensemble")
display(test_results)

test_preds = final_test_preds.numpy()
cm = confusion_matrix(test_targets.numpy(), test_preds, labels=range(5))
print("Rows = true [W, N1, N2, N3, REM]; columns = predicted stages")
print(cm)
ConfusionMatrixDisplay(cm, display_labels=stage_names).plot(
    cmap='Blues', values_format='d', colorbar=False,
)
plt.title(f"Held-out test - {K}-model soft-voting ensemble")
plt.tight_layout()
plt.show()

# Every artifact belongs to the selected RUN_DIR.
test_results.to_csv(RUN_DIR / "test_metrics.csv")
save_predictions(RUN_DIR / "test_predictions.csv", test_dataset, np.arange(len(test_dataset)),
                 test_targets.numpy(), mean_test_probs)
np.savetxt(RUN_DIR / "test_confusion_matrix.csv", cm, delimiter=",", fmt="%d")
run_metadata = json.loads((RUN_DIR / "run.json").read_text())
run_metadata["status"] = "evaluated"
run_metadata["test_ensemble_folds"] = K
save_json(RUN_DIR / "run.json", run_metadata)
